# Introduction

**Do checkout the THEORY Notebook:**

**https://www.kaggle.com/code/aryashah2k/introduction-to-it-ops-and-ai-theory**

**FAQ Recommendation For HelpDesk:**

**https://www.kaggle.com/code/aryashah2k/faq-recommendation-for-helpdesk**

**All Notebooks are under the Code Section of the dataset:**

**https://www.kaggle.com/datasets/aryashah2k/datasets-in-it-ops-applied-ai**

### What is Root Cause Analysis

RCA stands for root cause analysis, and it's pretty much exactly what it sounds like. It is an iterative, interrogative technique used to explore the cause-and-effect relationships of an underlying problem. 

When a problem happens, what we get to see are the symptoms of the problem. The symptoms need to be mapped to a root cause by asking a series of questions and conducting a set of exploratory tests. This is similar to how a doctor diagnoses a symptom, like a fever, to a root cause, like a viral infection. 

How does this apply to ITOps? 

ITOps receives a number of service incidents every day from users. The incidents usually state the symptoms that are observed by the user. ITOps engineers may further analyze the problem to identify more symptoms. Then comes the analysis process of narrowing down the symptom to its root cause. 

For example, the symptom might be a user seeing a message like, "We are unable to save your changes. "Please contact the administrator." The root cause might be that there is no disk space in the database. The time taken to narrow down the root cause is a significant factor in determining resolution times. It requires expert help, usually the developers of the software or the vendor who supplied the product. That expertise may take some time to come by. AI can help here by looking at the symptoms and predicting the root causes. 

This helps ITOps to get down to fixing the root cause quickly. In this chapter, we will implement such a use case.

### Classification With Deep Learning

Let us quickly review the concept of classification in machine learning and its implementation in deep learning. Classification is a machine learning problem of identifying a set of categories to which a new observation belongs based on a training set that contains observations for which the category is already known. 

Classification is the most common type of machine learning problem. Examples include binary classification, like if a customer would buy a product or not, or multiclass classification, like determining the type of customer based on customer demographics. There are a number of algorithms used for classification. 

Those include simple decision trees, naive Bayes, random forests, support vector machines, and deep learning. There are also a number of libraries that provide implementation of these algorithms. In Python, the most popular one is scikit-learn. In recent years, deep learning has revolutionized classification with its ability to handle complex relationships and generating highly accurate models.

We will implement root cause analysis using deep learning. We will use Keras library with TensorFlow backend for building the classification model.

### About The Dataset

I will review the input data we will use for building an RCA model. The input data is available in the file named root_cause_analysis.csv in the exercise folder. This data is collected about the symptoms seen for the problems reported. These symptoms are gathered by ITOps engineers when a problem is seen by the users. To keep it simple, each symptom is a separate attribute. That is a flag with values one or zero to indicate if the symptom is seen or not. The symptoms are CPU load, memory load, network delays, and whether some of these specific error codes were seen in the log files. the root cause attribute is the target variable that contains classes of root causes. 

This example uses a simple set of symptoms but in real-life scenarios there can be a large number of symptoms. Symptoms can further be broken down by where they are seen. Like for CPU load, was it at the load balancer or a specific microservice? Symptoms can also be isolated from general text using text mining. Let's now proceed to build a model based on this data set to predict the root cause based on the symptoms seen for the problem.

### Preprocessing Dataset

we will load up the RCA data set and pre-process it to make it ready for machine learning consumption. Remember that the purpose of doing all this is to predict the root cause based on the symptoms seen for the problem. 

Before we proceed, let's run the package setup to make sure that all packages required for this exercise are installed. We will use Pandas to load up the data set. 

First we load the CSV into a Pandas data frame using the read_csv method. We then print the data types and contents of the data frame to check if the loading happened successfully. 

Next, we need to convert data to formats that can be consumed by Keras as Keras only consumes NumPy arrays. First, the root cause column is a text attribute. We need to convert it into a numeric value. We will use the label encoder from Scikit-learn to transform the root cause into a numeric value. 

Next, we need to convert the data frame into a NumPy array using the to_numpy function. We then separate the training attributes X_train into the X_train array. For the target variable, we extract it to Y_train. Then we need to use the one-hot encoding on this categorical target attribute for it to be consumed by Keras. For this, we use the utils.to_categorical function. 

Finally, we print the shape of these variables to check if all the processing went through fine. Let's execute this code and check the results. We see the shapes printed correctly showing the thousand samples that are in the data set.

**Incident Reports in ITOps usually states the symptoms. Identifying the root cause of the symptom quickly is a key determinant to reducing resolution times and improving user satisfaction.**

In [1]:
# Importing necessary libraries
import pandas as pd
import os
import tensorflow as tf

#Load the data file into a Pandas Dataframe
symptom_data = pd.read_csv("../input/datasets-in-it-ops-applied-ai/root_cause_analysis.csv")

#Explore the data loaded
print(symptom_data.dtypes)
symptom_data.head()

ID              int64
CPU_LOAD        int64
MEMORY_LOAD     int64
DELAY           int64
ERROR_1000      int64
ERROR_1001      int64
ERROR_1002      int64
ERROR_1003      int64
ROOT_CAUSE     object
dtype: object


,ID,CPU_LOAD,MEMORY_LOAD,DELAY,ERROR_1000,ERROR_1001,ERROR_1002,ERROR_1003,ROOT_CAUSE
0,1,0,0,0,0,1,0,1,MEMORY
1,2,0,0,0,0,0,0,1,MEMORY
2,3,0,1,1,0,0,1,1,MEMORY
3,4,0,1,0,1,1,0,1,MEMORY
4,5,1,1,0,1,0,1,0,NETWORK_DELAY


In [2]:
# Convert input data to formats that can be used by ML Algorithms
from sklearn import preprocessing

label_encoder = preprocessing.LabelEncoder()
symptom_data['ROOT_CAUSE'] = label_encoder.fit_transform(
                                symptom_data['ROOT_CAUSE'])

#Convert Pandas DataFrame to a numpy vector
np_symptom = symptom_data.to_numpy().astype(float)

#Extract the feature variables (X)
X_train = np_symptom[:,1:8]

#Extract the target variable (Y), convert to one-hot-encoding
Y_train=np_symptom[:,8]
Y_train = tf.keras.utils.to_categorical(Y_train,3)

print("Shape of feature variables :", X_train.shape)
print("Shape of target variable :",Y_train.shape)

Shape of feature variables : (1000, 7)
Shape of target variable : (1000, 3)


###  Building A Classification Model With Keras

Let's create a softmax model with three dense layers in Keras to predict the root cause based on the symptoms. it's highly recommended for the students to have prior understanding of how deep learning and Keras work for learning these examples. First, we set up the hyperparameters for the neural network. We set epochs to 20 and a batch size of hundred. 

Please note that we have thousand samples in the dataset, so there will be 10 batches overall. We set verbose to be one. So we can view the details of model training. The number of output classes will be set to match the unique number of labels in the target variable, namely root cause. We will then try with a hidden layer size of 128. For validation we split 20% of the thousand input records to be validation data. Now we can create a sequential model. We add the first hidden layer as a dense layer with activation as ReLU also called rectified linear unit. 

We add a second similar layer. Finally, we add a softmax layer to provide categorical labels. We then compelled the model using Adam optimizer and set the loss function as categorical, cross, and trophy. We then proceed to fit the model with X_train and Y_train in order to create the model. We print the model summary to see the model structure. Let's now, execute this code. On running this code, we see that the model can achieve an accuracy of 86% with the validation data. Now that we have the model set up and trained, let's use it to predict root causes.

In [3]:
from tensorflow import keras
from tensorflow.keras import optimizers
from tensorflow.keras.regularizers import l2

#Setup Training Parameters
EPOCHS=20
BATCH_SIZE=100
VERBOSE=1
OUTPUT_CLASSES=len(label_encoder.classes_)
N_HIDDEN=128
VALIDATION_SPLIT=0.2

#Create a Keras sequential model
model = tf.keras.models.Sequential()
#Add a Dense Layer
model.add(keras.layers.Dense(N_HIDDEN,
                             input_shape=(7,),
                              name='Dense-Layer-1',
                              activation='relu'))

#Add a second dense layer
model.add(keras.layers.Dense(N_HIDDEN,
                              name='Dense-Layer-2',
                              activation='relu'))

#Add a softmax layer for categorial prediction
model.add(keras.layers.Dense(OUTPUT_CLASSES,
                             name='Final',
                             activation='softmax'))

#Compile the model, using Adam optimizer
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

#Build the model
model.fit(X_train,
          Y_train,
          batch_size=BATCH_SIZE,
          epochs=EPOCHS,
          verbose=VERBOSE,
          validation_split=VALIDATION_SPLIT)

model.summary()

2022-06-09 10:38:39.547614: I tensorflow/core/common_runtime/process_util.cc:146] Creating new thread pool with default inter op setting: 2. Tune using inter_op_parallelism_threads for best performance.
2022-06-09 10:38:39.715669: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:185] None of the MLIR Optimization Passes are enabled (registered 2)


Epoch 1/20
8/8 [==============================] - 1s 33ms/step - loss: 1.0130 - accuracy: 0.6212 - val_loss: 0.9231 - val_accuracy: 0.7500
Epoch 2/20
8/8 [==============================] - 0s 7ms/step - loss: 0.8428 - accuracy: 0.7937 - val_loss: 0.7712 - val_accuracy: 0.8000
Epoch 3/20
8/8 [==============================] - 0s 7ms/step - loss: 0.6929 - accuracy: 0.8175 - val_loss: 0.6457 - val_accuracy: 0.7700
Epoch 4/20
8/8 [==============================] - 0s 8ms/step - loss: 0.5729 - accuracy: 0.8163 - val_loss: 0.5610 - val_accuracy: 0.7800
Epoch 5/20
8/8 [==============================] - 0s 7ms/step - loss: 0.4933 - accuracy: 0.8163 - val_loss: 0.5236 - val_accuracy: 0.8000
Epoch 6/20
8/8 [==============================] - 0s 7ms/step - loss: 0.4549 - accuracy: 0.8250 - val_loss: 0.5169 - val_accuracy: 0.8000
Epoch 7/20
8/8 [==============================] - 0s 7ms/step - loss: 0.4365 - accuracy: 0.8388 - val_loss: 0.5201 - val_accuracy: 0.8000
Epoch 8/20
8/8 [=================

### Predicting Root Causes

Now that we have built an RCA model, let's use it to predict the root cause for a new incident. When a new incident happen, we typically identify the symptoms of the incident first, and populate the related feature variables here, like CPU load, memory load, delays and error codes. 

We then pass these as an array to the model's predict classes function. This function will return a numeric value for the root cause. We then translate the numeric value into a label using the inverse transform function on the encoder.

As we can see, it predicts the root cause to be a database issue. It is possible to use this model to predict root causes for a batch of incidents. Batch predictions are a lot more efficient than executing them one by one. To do this, we create an array of arrays with the inside array, representing a row of feature variables. On calling the predict classes method on the model, we get a list of numeric predictions. 

We then pass this batch to the inverse transform method of the label encoder to get the list of root causes. Let's execute this code. We now see, the root causes printed in the same order as the rows in the input data. I highly recommend experimenting with the hyperparameters of this model, as well as modifying the hidden layers to arrive at the best combination of accuracy and performance for the model. When you try the same with your own data, this tuning is required to arrive at an optimal balance of speed and accuracy.

In [11]:
#Pass individual flags to Predict the root cause
import numpy as np
CPU_LOAD=1
MEMORY_LOAD=0
DELAY=0
ERROR_1000=0
ERROR_1001=1
ERROR_1002=1
ERROR_1003=0

prediction=np.argmax(model.predict(
    [[CPU_LOAD,MEMORY_LOAD,DELAY,
      ERROR_1000,ERROR_1001,ERROR_1002,ERROR_1003]]), axis=1)

print(label_encoder.inverse_transform(prediction))

['DATABASE_ISSUE']


In [21]:
#Predicting as a Batch
batch_prediction = np.argmax(model.predict([[1,0,0,0,1,1,0],
                                [0,1,1,1,0,0,0],
                                [1,1,0,1,1,0,1],
                                [0,0,0,0,0,1,0],
                                [1,0,1,0,1,1,1]]), axis =1)

In [22]:
print(label_encoder.inverse_transform(batch_prediction))

['DATABASE_ISSUE' 'NETWORK_DELAY' 'MEMORY' 'DATABASE_ISSUE'
 'DATABASE_ISSUE']


### Conclusion

Witht his we conclude the notebook after successfully applying AI to a use-case of Incident Root Cause Analysis in IT Ops.

⭐ Have a great day! ⭐

📙📙📙📙📙📙📙📙📙📙 Happy Learning! 📙📙📙📙📙📙📙📙📙📙